# Regularization for Deep Learning — Notebook 3 of 3
## Constraints · Reprojection & Max-Norm · Under-Constrained Problems

**Companion to Chapter 7, Stations 6–8.**  The finale. Two big ideas:

> **(a)** Every norm penalty is secretly a **hard constraint** — "keep the weights inside a region `Ω(w) < k`." Bigger `α` = smaller region.
>
> **(b)** Some problems have **no unique answer** without regularization. A dab of `αI` makes a singular matrix invertible and stops separable logistic regression from running to infinity.

### What you will do here
1. **Penalty ⇄ constraint** — measure the implied region size `k` as `α` grows.
2. **Reprojection** — projected gradient descent onto an L² ball.
3. **Max-norm** — cap each hidden unit's incoming weights (Hinton's dropout companion).
4. **Singular `XᵀX`** — watch `+αI` restore invertibility and crush the condition number.
5. **The separable-data trap** — logistic weights → ∞ without decay, finite with it.
6. **The pseudoinverse** — the `α → 0⁺` limit of ridge.

Run top to bottom. **🔧 Try it** cells are editable; **✏️ Exercise** cells are yours.

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

C_DATA = "#0F7C87"; C_L2 = "#6C30D9"; C_L1 = "#C81C6E"; C_WARN = "#B27412"
print("Ready.")

## 1 · Penalty ⇄ constraint — two faces of one problem

Minimizing `J(θ)` subject to `Ω(θ) < k` has the generalized Lagrangian

$$\mathcal L(\theta,\alpha)=J(\theta)+\alpha\,(\Omega(\theta)-k),\qquad \theta^*=\arg\min_\theta\;\max_{\alpha\ge 0}\;\mathcal L(\theta,\alpha)$$

Fix the optimal `α` and this is *exactly* the penalized objective `J + αΩ` from Notebook 1. So:

* **α and k move oppositely.** A larger `α` forces a smaller region `k`; weights end up nearer the origin.
* **k is implicit.** You rarely know the exact `k` a given `α` implies — but you always know the direction.

Let's make `k` visible: for a 2-D ridge problem, solve at each `α` and read off `k = ‖w̃‖²` (the region the solution actually lands in).

In [ ]:
# A simple 2-D least-squares problem: minimize ½‖Aw − b‖²  → optimum w* = A⁻¹b
A = np.array([[3.0, 0.5], [0.5, 1.0]])
b = np.array([4.0, 2.0])
H = A.T @ A                                  # Hessian of ½‖Aw−b‖²
g = A.T @ b
w_star = np.linalg.solve(H, g)

def w_ridge(alpha):
    return np.linalg.solve(H + alpha * np.eye(2), g)

alphas = np.linspace(0, 40, 200)
norms = np.array([np.linalg.norm(w_ridge(a)) for a in alphas])

plt.figure()
plt.plot(alphas, norms, color=C_L2, lw=2)
plt.axhline(np.linalg.norm(w_star), color=C_DATA, ls="--", label=f"‖w*‖ = {np.linalg.norm(w_star):.2f}")
plt.xlabel("regularization α"); plt.ylabel("‖w̃‖   ( = radius of the region k )")
plt.title("Turning α up shrinks the ball the weights may live in")
plt.legend(); plt.show()

print("Read it as a budget: bigger α ⇒ smaller k ⇒ weights confined nearer the origin.")
print(f"α=0 → ‖w̃‖={norms[0]:.2f}   α=40 → ‖w̃‖={norms[-1]:.2f}")

**✏️ Exercise 1.** Verify the equivalence directly. Pick `α = 5`, compute `w_ridge(5)` and its `k = ‖w̃‖`. Then solve the *constrained* problem "minimize `½‖Aw−b‖²` s.t. `‖w‖ ≤ k`" numerically (e.g. `scipy.optimize.minimize` with a constraint, or project-and-check) and confirm you land on the **same** `w̃`. Penalty and constraint are the same solution seen through different knobs.

In [ ]:
# ✏️ Your code here.


## 2 · Reprojection — projected gradient descent

Take the constraint *literally*: after each gradient step, if the weights leave the region, **project** them back onto its boundary.

```
w ← w − ε·∇J(w)                         # ordinary step
if ‖w‖ > k:  w ← w · (k / ‖w‖)          # snap back onto the ball
```

The weights ride the boundary instead of being dragged toward the origin. Below we run PGD onto an L² ball and draw the trajectory over the loss contours.

In [ ]:
def project_l2(w, k):
    nrm = np.linalg.norm(w)
    return w * (k / nrm) if nrm > k else w

def pgd(k, eps=0.03, steps=60, w0=np.array([0.2, 0.2])):
    w, path = w0.copy(), [w0.copy()]
    for _ in range(steps):
        w = w - eps * (H @ w - g)          # gradient of ½‖Aw−b‖²
        w = project_l2(w, k)
        path.append(w.copy())
    return np.array(path)

k = 1.2
path = pgd(k)

gx = np.linspace(-0.5, 2.2, 300); gy = np.linspace(-0.5, 2.2, 300)
GX, GY = np.meshgrid(gx, gy)
P = np.stack([GX, GY], -1)
LOSS = 0.5 * np.einsum("...i,ij,...j->...", P, H, P) - P @ g   # ½ wᵀHw − wᵀg

plt.figure(figsize=(6.2, 6))
plt.contour(GX, GY, LOSS, levels=25, colors=C_DATA, alpha=0.4, linewidths=0.7)
th = np.linspace(0, 2*np.pi, 300)
plt.plot(k*np.cos(th), k*np.sin(th), color=C_L2, lw=2, label=f"‖w‖ ≤ k = {k}")
plt.plot(path[:, 0], path[:, 1], color=C_L1, marker="o", ms=3, lw=1.4, label="PGD trajectory")
plt.scatter(*w_star, color=C_DATA, s=70, zorder=5, label="unconstrained w*")
plt.scatter(*path[-1], color=C_L1, s=70, zorder=6, label="constrained w̃")
plt.gca().set_aspect("equal"); plt.legend(loc="upper left")
plt.title("Projected gradient descent: step freely, then snap back to the ball")
plt.show()

print(f"final ‖w̃‖ = {np.linalg.norm(path[-1]):.3f}  (rides the boundary k = {k})")

**Why reprojection can beat a penalty**

* **No dead zones.** A penalty can create flat regions near the origin where non-convex optimization stalls with all-tiny weights. Reprojection never *encourages* weights toward 0, so it avoids those dead spots.
* **Stability.** Penalties can trigger runaway feedback (big weights → big gradients → bigger weights). Reprojection caps the norm regardless of learning rate.
* **Per-column max-norm** (next).

## 3 · Max-norm — cap each unit separately

Max-norm constrains the norm of **each column** of a weight matrix (each hidden unit's incoming weights) to `≤ c`, so no single unit can shout much louder than the others. It's the constraint Hinton et al. paired with dropout to allow aggressive learning rates.

```python
# after optimizer.step():
norms = weights.norm(dim=0, keepdim=True)   # per-column norm
scale = (c / norms).clamp(max=1.0)          # only shrink, never grow
weights *= scale                            # reproject each column
```

Let's visualize the reprojection on a random weight matrix, then run it inside a real PyTorch training loop.

In [ ]:
# Pure-NumPy illustration of per-column max-norm reprojection.
rng = np.random.default_rng(2)
W = rng.normal(0, 1.2, size=(10, 12))          # 10 inputs × 12 hidden units
c = 3.0
col_norms = np.linalg.norm(W, axis=0)
scale = np.minimum(1.0, c / col_norms)
W_capped = W * scale
new_norms = np.linalg.norm(W_capped, axis=0)

x = np.arange(W.shape[1])
plt.figure()
plt.bar(x - 0.2, col_norms, width=0.4, color=C_WARN, label="before")
plt.bar(x + 0.2, new_norms, width=0.4, color=C_L2, label="after max-norm")
plt.axhline(c, color=C_L1, ls="--", label=f"cap c = {c}")
plt.xlabel("hidden unit (column)"); plt.ylabel("incoming weight norm")
plt.title("Max-norm rescales only the units that exceed the cap"); plt.legend(); plt.show()

print("Units under the cap are untouched; units above it are pulled down exactly to c.")

**🔧 Try it — max-norm in a PyTorch training loop.** We apply the per-column cap after every optimizer step and compare against an unconstrained run.

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
Xm, ym = make_moons(n_samples=200, noise=0.30, random_state=0)
Xt, Xv, yt, yv = train_test_split(Xm, ym, test_size=0.5, random_state=0)
tt = lambda a: torch.tensor(a, dtype=torch.float32)
Xt, Xv, yt, yv = tt(Xt), tt(Xv), tt(yt), tt(yv)

def make_net():
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 1))

def apply_max_norm(net, c):
    with torch.no_grad():
        for name, p in net.named_parameters():
            if "weight" in name:
                norms = p.norm(dim=1, keepdim=True)          # per-output-unit
                p.mul_((c / norms).clamp(max=1.0))

def train(max_norm_c=None, epochs=400, lr=0.5):     # high lr on purpose
    net = make_net()
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    wnorm, vacc = [], []
    for _ in range(epochs):
        opt.zero_grad()
        lossf(net(Xt).squeeze(), yt).backward(); opt.step()
        if max_norm_c is not None:
            apply_max_norm(net, max_norm_c)
        with torch.no_grad():
            wnorm.append(sum(p.pow(2).sum() for n,p in net.named_parameters() if "weight" in n).sqrt().item())
            vacc.append((((net(Xv).squeeze() > 0).float() == yv).float().mean()).item())
    return wnorm, vacc

wn_free, va_free = train(max_norm_c=None)
wn_mn,   va_mn   = train(max_norm_c=2.0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(wn_free, color=C_WARN, label="unconstrained")
ax[0].plot(wn_mn,   color=C_L2,   label="max-norm c=2")
ax[0].set_title("Total ‖w‖ (high learning rate)"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(va_free, color=C_WARN, label="unconstrained")
ax[1].plot(va_mn,   color=C_L2,   label="max-norm c=2")
ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Max-norm keeps the weight norm bounded even with an aggressive learning rate.")

## 4 · Under-constrained problems — `+αI` to the rescue

Linear regression, PCA and friends invert `XᵀX`. That matrix is **singular** when the data has no variance in some direction, or when `m < n` (fewer samples than features). Then `(XᵀX)⁻¹` doesn't exist and the optimum runs off to infinity.

$$(X^\top X)^{-1}\text{ may not exist}\;\longrightarrow\;(X^\top X+\alpha I)^{-1}\text{ always exists }(\alpha>0)$$

Adding `α` to every diagonal entry lifts every eigenvalue by `α`, so the smallest can no longer be zero. Let's build a rank-deficient `X` and watch the condition number.

In [ ]:
# 30 samples, 8 features, but the feature matrix is rank-deficient (a column is a copy).
rng = np.random.default_rng(5)
Xr = rng.normal(size=(30, 8))
Xr[:, 3] = Xr[:, 1]                 # exact collinearity -> XᵀX singular
XtX = Xr.T @ Xr
eig = np.linalg.eigvalsh(XtX)
print("smallest eigenvalue of XᵀX:", eig.min().round(6), " (≈0 ⇒ singular ⇒ not invertible)")

alphas = np.logspace(-4, 2, 200)
lam_max = eig.max()
lam_min = eig.min()
cond = (lam_max + alphas) / (lam_min + alphas)

plt.figure()
plt.plot(alphas, cond, color=C_L2, lw=2)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("regularization α"); plt.ylabel("condition number κ = (λmax+α)/(λmin+α)")
plt.title("α trades a little bias for a lot of numerical stability")
plt.show()

# Demonstrate invertibility flips on:
def invertible(M):
    try:
        np.linalg.inv(M); return True
    except np.linalg.LinAlgError:
        return False
print("invertible XᵀX      :", invertible(XtX))
print("invertible XᵀX + αI :", invertible(XtX + 1e-2 * np.eye(8)), "(α = 1e-2)")

## 5 · The separable-data trap — weights to infinity

There's a sneakier under-constrained case. If a logistic-regression boundary **perfectly separates** two classes, then doubling the weights (`2w`) always raises the likelihood — and so on forever. Without regularization `‖w‖ → ∞`; the optimum sits at infinity.

Any weight decay halts the runaway at a finite point where the likelihood gain finally equals the decay cost. Watch `‖w‖` over training with `α = 0` vs `α > 0`.

In [ ]:
# Perfectly separable 2-D data.
rng = np.random.default_rng(7)
n = 40
Xpos = rng.normal([ 2.0, 2.0], 0.4, size=(n, 2))
Xneg = rng.normal([-2.0,-2.0], 0.4, size=(n, 2))
Xs = np.vstack([Xpos, Xneg]); ys = np.hstack([np.ones(n), np.zeros(n)])

def sigmoid(z): return 1 / (1 + np.exp(-z))

def train_logreg(alpha, steps=1500, eps=0.1):
    w = np.zeros(2); b = 0.0; norms = []
    for _ in range(steps):
        z = Xs @ w + b
        p = sigmoid(z)
        grad_w = Xs.T @ (p - ys) / len(ys) + alpha * w   # + weight decay on w
        grad_b = np.mean(p - ys)                          # bias unregularized
        w -= eps * grad_w; b -= eps * grad_b
        norms.append(np.linalg.norm(w))
    return w, b, np.array(norms)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
# left: the data + a boundary
_, _, _ = train_logreg(0.0)
ax[0].scatter(Xpos[:,0], Xpos[:,1], color=C_DATA, s=18, label="class 1")
ax[0].scatter(Xneg[:,0], Xneg[:,1], color=C_L1,   s=18, label="class 0")
ax[0].set_title("Perfectly separable classes"); ax[0].legend(); ax[0].set_aspect("equal")

# right: ‖w‖ over steps for several α
for a, col in [(0.0, C_WARN), (0.01, C_L2), (0.05, C_L1)]:
    _, _, nrm = train_logreg(a)
    ax[1].plot(nrm, color=col, label=f"α = {a}")
ax[1].set_title("‖w‖ over 1500 steps"); ax[1].set_xlabel("step"); ax[1].set_ylabel("‖w‖")
ax[1].legend()
plt.tight_layout(); plt.show()

print("α = 0 (amber): ‖w‖ keeps climbing — no finite optimum.")
print("α > 0        : the curve bends to a plateau — the problem is now well-posed.")

**✏️ Exercise 5.** scikit-learn's `LogisticRegression` is L²-regularized *by default* (its `C` parameter is the **inverse** strength, so small `C` = strong regularization) precisely to stop this divergence. Fit it on `(Xs, ys)` for `C in [0.01, 1, 100, 10000]` and print `‖coef_‖` for each. Confirm large `C` (weak regularization) gives large weights.

In [ ]:
# ✏️ Your code here.
# from sklearn.linear_model import LogisticRegression
# for C in [0.01, 1, 100, 10000]:
#     clf = LogisticRegression(C=C, penalty='l2').fit(Xs, ys)
#     print(C, np.linalg.norm(clf.coef_))


## 6 · The pseudoinverse is the `α → 0⁺` limit of ridge

The Moore–Penrose pseudoinverse — the tool that gives a stable answer to under-determined systems — is exactly the vanishing-regularization limit of ridge regression:

$$X^{+}=\lim_{\alpha\to 0^{+}}(X^\top X+\alpha I)^{-1}X^\top$$

Read it as: regularize, then let the leash go slack. What remains is the **minimum-norm** solution — the well-behaved answer regularization was quietly selecting all along.

In [ ]:
# Under-determined system: fewer samples than features (m < n) -> infinitely many exact fits.
rng = np.random.default_rng(11)
m, nfeat = 5, 12
Xu = rng.normal(size=(m, nfeat))
yu = rng.normal(size=m)

pinv_sol = np.linalg.pinv(Xu) @ yu               # minimum-norm solution

alphas = np.logspace(-6, 1, 30)
diffs = []
for a in alphas:
    ridge_sol = np.linalg.solve(Xu.T @ Xu + a * np.eye(nfeat), Xu.T @ yu)
    diffs.append(np.linalg.norm(ridge_sol - pinv_sol))

plt.figure()
plt.plot(alphas, diffs, color=C_L2, marker="o", ms=3)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("α"); plt.ylabel("‖ ridge(α) − pseudoinverse ‖")
plt.title("Ridge → pseudoinverse as α → 0⁺")
plt.show()

print("min-norm ‖X⁺y‖ :", round(float(np.linalg.norm(pinv_sol)), 4))
print("ridge(α=1e-6) matches the pseudoinverse to:",
      f"{diffs[0]:.2e}")
# All ridge solutions still fit the data exactly? Check residual at tiny α:
tiny = np.linalg.solve(Xu.T @ Xu + 1e-6 * np.eye(nfeat), Xu.T @ yu)
print("training residual at α=1e-6:", round(float(np.linalg.norm(Xu @ tiny - yu)), 4),
      "(≈0 ⇒ still an exact fit, just the smallest-norm one)")

## Key takeaways — and the whole course in one picture

1. **Penalty ⇄ constraint.** `J + α(Ω − k)`: a penalty of strength `α` equals a constraint of some region size `k`. More `α` ⇒ smaller region ⇒ weights nearer the origin.
2. **Explicit constraints are steadier.** Reproject after each step: no dead zones near the origin, no runaway divergence, and per-column **max-norm** for stable high-learning-rate training.
3. **Regularization makes problems well-posed.** `XᵀX + αI` is always invertible; separable logistic regression gets a finite solution; the pseudoinverse is the `α → 0⁺` limit of ridge.

> **If you remember only one thing (all three notebooks):** a norm penalty is a **second force** on the weights. The data term pulls toward the fit `w*`; the penalty pulls toward `0`; the solution `w̃` is the balance. Everything — weight decay, ridge, LASSO, constraint balls, reprojection, the pseudoinverse — is just **which shape of pull** you choose and **how hard** you pull.

### Where each idea shows up in practice
- **Weight decay / AdamW** — the default dial in every deep-learning optimizer.
- **Ridge** — correlated features in finance, econometrics, genomics.
- **LASSO / Elastic Net** — feature selection, sparse coding, network pruning.
- **Max-norm & gradient clipping** — stabilizing dropout nets, RNNs, Transformers, GANs, RL.
- **`+αI` / pseudoinverse** — the `p ≫ n` regime: text, genomics, tiny fine-tuning sets.

*You've walked all eight stations. Re-run the labs before the exam.*